In [ ]:
# --- INSTALL FLOP COUNTER ---
# We use 'thop' to calculate FLOPs
!pip install thop

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time
from thop import profile # Library for FLOPs

# --- CONFIGURATION ---
BATCH_SIZE = 64
EPOCHS = 1      # 1 Epoch is enough to measure speed difference

# --- PREPARE DATA (Full FashionMNIST) ---
print("Preparing Data...")
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Full Dataset (Standard Split: 60k Train, 10k Test)
train_set = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# drop_last=True prevents error if the very last batch has size 1
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, drop_last=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# --- MODEL SETUP ---
def get_model():
    model = torchvision.models.resnet18(pretrained=False)
    # Adjust for 1-channel FashionMNIST
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    return model

# --- TRAINING FUNCTION ---
def run_training(device_type):
    device = torch.device(device_type)
    model = get_model().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    print(f"\n--- STARTING TRAINING ON {device_type.upper()} ---")

    # Warmup (CRITICAL FIX: Batch size must be > 1 for BatchNorm)
    if device_type == 'cuda':
        dummy_input = torch.randn(2, 1, 32, 32).to(device) # Batch size 2 prevents ValueError
        model(dummy_input)
        torch.cuda.synchronize()

    start_time = time.time()

    model.train()
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # Print progress every 100 batches
        if i % 100 == 0:
            print(f"  [Batch {i}/{len(train_loader)}] Processing...")

    if device_type == 'cuda':
        torch.cuda.synchronize()

    end_time = time.time()
    total_time = end_time - start_time

    # Calculate Accuracy
    print(f"  Training finished. Evaluating accuracy...")
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = 100 * correct / total

    print(f"RESULT ({device_type.upper()}): Time {total_time:.2f}s | Acc {acc:.2f}%")
    return total_time, acc

# --- 1. CALCULATE FLOPs ---
print("\n--- CALCULATING FLOPs ---")
try:
    model_flop = get_model()
    input_dummy = torch.randn(1, 1, 32, 32)
    flops, params = profile(model_flop, inputs=(input_dummy, ), verbose=False)
    print(f"FLOPs: {flops/1e9:.4f} GFLOPs")
    print(f"Params: {params/1e6:.4f} Million")
except Exception as e:
    print(f"FLOP calculation warning: {e}")

# --- 2. RUN EXPERIMENTS ---
# Run GPU first (Fast)
if torch.cuda.is_available():
    gpu_time, gpu_acc = run_training('cuda')
else:
    print("Error: GPU not available. Make sure you set Runtime > Change runtime type > T4 GPU")
    gpu_time = 0

# Run CPU second (Slow - might take 3-5 mins)
print("\n(Note: CPU training is slow. Please wait...)")
cpu_time, cpu_acc = run_training('cpu')

# --- 3. SUMMARY ---
print("\n" + "="*30)
print("FINAL COMPARISON REPORT")
print("="*30)
print(f"1. FLOPs per Image: {flops/1e9:.4f} GFLOPs")
print(f"2. GPU Time: {gpu_time:.2f}s")
print(f"3. CPU Time: {cpu_time:.2f}s")
if gpu_time > 0:
    print(f"4. Speedup: {cpu_time / gpu_time:.2f}x faster on GPU")

Preparing Data...

--- CALCULATING FLOPs ---
FLOPs: 0.0361 GFLOPs
Params: 11.6832 Million

--- STARTING TRAINING ON CUDA ---
  [Batch 0/937] Processing...
  [Batch 100/937] Processing...
  [Batch 200/937] Processing...
  [Batch 300/937] Processing...
  [Batch 400/937] Processing...
  [Batch 500/937] Processing...
  [Batch 600/937] Processing...
  [Batch 700/937] Processing...
  [Batch 800/937] Processing...
  [Batch 900/937] Processing...
  Training finished. Evaluating accuracy...
RESULT (CUDA): Time 27.90s | Acc 86.88%

(Note: CPU training is slow. Please wait...)

--- STARTING TRAINING ON CPU ---
  [Batch 0/937] Processing...
  [Batch 100/937] Processing...
  [Batch 200/937] Processing...
  [Batch 300/937] Processing...
  [Batch 400/937] Processing...
  [Batch 500/937] Processing...
  [Batch 600/937] Processing...
  [Batch 700/937] Processing...
  [Batch 800/937] Processing...
  [Batch 900/937] Processing...
  Training finished. Evaluating accuracy...
RESULT (CPU): Time 738.41s | Ac

In [ ]:

    # --- INSTALL FLOP COUNTER ---
# We use 'thop' to calculate FLOPs
!pip install thop

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time
from thop import profile # Library for FLOPs

# --- CONFIGURATION ---
BATCH_SIZE = 64
EPOCHS = 1      # 1 Epoch is enough to measure speed difference

# --- PREPARE DATA (Full FashionMNIST) ---
print("Preparing Data...")
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Full Dataset (Standard Split: 60k Train, 10k Test)
train_set = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# drop_last=True prevents error if the very last batch has size 1
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, drop_last=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)


# --- MODEL SETUP (ResNet-50) ---
def get_model():
    # 1. Load ResNet-50 instead of ResNet-18
    model = torchvision.models.resnet50(pretrained=False)

    # 2. Adjust First Layer: 1-channel input (Grayscale) instead of 3 (RGB)
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

    # 3. Adjust Last Layer: 10 Output Classes (for FashionMNIST)
    # ResNet-50's FC layer has 2048 input features (vs 512 in ResNet-18)
    model.fc = nn.Linear(model.fc.in_features, 10)

    return model



# --- TRAINING FUNCTION ---
def run_training(device_type):
    device = torch.device(device_type)
    model = get_model().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    print(f"\n--- STARTING TRAINING ON {device_type.upper()} ---")

    # Warmup (CRITICAL FIX: Batch size must be > 1 for BatchNorm)
    if device_type == 'cuda':
        dummy_input = torch.randn(2, 1, 32, 32).to(device) # Batch size 2 prevents ValueError
        model(dummy_input)
        torch.cuda.synchronize()

    start_time = time.time()

    model.train()
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # Print progress every 100 batches
        if i % 100 == 0:
            print(f"  [Batch {i}/{len(train_loader)}] Processing...")

    if device_type == 'cuda':
        torch.cuda.synchronize()

    end_time = time.time()
    total_time = end_time - start_time

    # Calculate Accuracy
    print(f"  Training finished. Evaluating accuracy...")
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = 100 * correct / total

    print(f"RESULT ({device_type.upper()}): Time {total_time:.2f}s | Acc {acc:.2f}%")
    return total_time, acc

# --- 1. CALCULATE FLOPs ---
print("\n--- CALCULATING FLOPs ---")
try:
    model_flop = get_model()
    input_dummy = torch.randn(1, 1, 32, 32)
    flops, params = profile(model_flop, inputs=(input_dummy, ), verbose=False)
    print(f"FLOPs: {flops/1e9:.4f} GFLOPs")
    print(f"Params: {params/1e6:.4f} Million")
except Exception as e:
    print(f"FLOP calculation warning: {e}")

# --- 2. RUN EXPERIMENTS ---
# Run GPU first (Fast)
if torch.cuda.is_available():
    gpu_time, gpu_acc = run_training('cuda')
else:
    print("Error: GPU not available. Make sure you set Runtime > Change runtime type > T4 GPU")
    gpu_time = 0

# Run CPU second (Slow - might take 3-5 mins)
print("\n(Note: CPU training is slow. Please wait...)")
cpu_time, cpu_acc = run_training('cpu')

# --- 3. SUMMARY ---
print("\n" + "="*30)
print("FINAL COMPARISON REPORT")
print("="*30)
print(f"1. FLOPs per Image: {flops/1e9:.4f} GFLOPs")
print(f"2. GPU Time: {gpu_time:.2f}s")
print(f"3. CPU Time: {cpu_time:.2f}s")
if gpu_time > 0:
    print(f"4. Speedup: {cpu_time / gpu_time:.2f}x faster on GPU")